# Production Run: 8,500-14,000 Examples

Full production generation across all 5 curriculum tiers, followed by
evaluation, quality filtering, OSFT validation, and dual-format serialization.

**Prerequisites:**
- D-19 dry-run validated by Farbod's team (do NOT run this until confirmed)
- vLLM serving Gemma 4 31B (`VLLM_API_KEY`, `VLLM_API_BASE`)
- Langflow agent deployed (`LANGFLOW_API_KEY`, `LANGFLOW_URL`)
- MCP Gateway reachable (`GATEWAY_URL`, `KEYCLOAK_URL`, `CLIENT_SECRET`)
- Seeds materialized (this notebook does it automatically if needed)
- This notebook is intended to run in an RHOAI workbench pointed at the `OTel_agentic_improvements` repo

**Target:**
- Raw: 8,500-14,000 examples across 5 tiers
- Filtered: 5,000-10,000 high-quality examples (SDG-04)
- Output: dual-format OSFT JSONL (single-turn + multi-turn) in sdg/data/delivery/

In [ ]:
import sys
import os
import json
import time
import tempfile
import logging
from pathlib import Path

sys.path.insert(0, os.path.abspath('../..'))

try:
    import yaml
except ImportError:
    !pip install pyyaml
    import yaml

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

In [ ]:
# ── Config Loading with env-var interpolation ──

def _expand_vars_recursive(obj):
    """Recursively apply os.path.expandvars to all string values."""
    if isinstance(obj, str):
        expanded = os.path.expandvars(obj)
        if '${' in expanded:
            logger.warning('Unexpanded env var: %r -- set the variable', expanded)
        return expanded
    elif isinstance(obj, dict):
        return {k: _expand_vars_recursive(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [_expand_vars_recursive(item) for item in obj]
    return obj

def _load_config(path):
    """Load YAML config with os.path.expandvars() interpolation."""
    with open(path) as f:
        raw = yaml.safe_load(f)
    return _expand_vars_recursive(raw) if raw else {}

# Load production config
config = _load_config('../../sdg/config/flow-training.yaml')
eval_config = _load_config('../../sdg/config/flow-evaluation.yaml')

# Production runtime params: 20x multiplication, 3-tool subsets
# These match flow-training.yaml defaults for production volume
production_params = {
    'multiply_tool_rows': {'num_samples': 20},
    'sample_tools': {'num_samples': 3},
}

print(f'Model: {config.get("model_config", {}).get("model")}')
print(f'Runtime params: {production_params}')

In [ ]:
# ── Seed Materialization ──
# Ensure tool metadata seeds are materialized from the MCP Gateway

from sdg.scripts.materialize_seeds import (
    materialize_tool_seeds,
    acquire_jwt,
    mcp_initialize,
    fetch_tools,
)

seeds_output = Path('../../sdg/data/seeds/tool_metadata.jsonl')
seeds_config_path = '../../sdg/config/seed-scenarios.yaml'

need_materialize = True
if seeds_output.exists():
    age = time.time() - seeds_output.stat().st_mtime
    if age < 3600:
        count = sum(1 for line in seeds_output.open() if line.strip())
        print(f'Seeds file is fresh ({age:.0f}s old, {count} records) -- skipping')
        need_materialize = False

if need_materialize:
    gateway_url = os.environ.get('GATEWAY_URL', 'https://mcp.apps.cluster-wfhjs.wfhjs.sandbox1330.opentlc.com/mcp')
    keycloak_url = os.environ.get('KEYCLOAK_URL', 'https://mcp-keycloak-otel25-gateway.apps.cluster-wfhjs.wfhjs.sandbox1330.opentlc.com')
    client_id = os.environ.get('CLIENT_ID', 'sdg-pipeline')
    client_secret = os.environ.get('CLIENT_SECRET', '')
    if not client_secret:
        raise ValueError('CLIENT_SECRET env var required')
    jwt = acquire_jwt(keycloak_url, client_id, client_secret)
    session_id = mcp_initialize(gateway_url, jwt)
    tools = fetch_tools(gateway_url, jwt, session_id)
    count = materialize_tool_seeds(tools, seeds_config_path, str(seeds_output))
    print(f'Materialized {count} seed records from {len(tools)} gateway tools')

In [ ]:
# ── Load All 5 Seed Tiers ──

seeds_by_tier = {}
with open(seeds_output) as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        record = json.loads(line)
        tier = record.get('tier', 'unknown')
        seeds_by_tier.setdefault(tier, []).append(record)

print('Seed distribution:')
total_seeds = 0
for tier, seeds in sorted(seeds_by_tier.items()):
    print(f'  {tier}: {len(seeds)} seeds (complexity {seeds[0].get("complexity", "?")})')
    total_seeds += len(seeds)
print(f'Total seeds: {total_seeds}')

In [ ]:
# ── Generation with Progress Tracking and Resume ──

from sdg_hub import FlowRegistry, Flow

FlowRegistry.discover_flows()
flow_path = FlowRegistry.get_flow_path('MCP Server Distillation')
flow = Flow.from_yaml(flow_path)

output_dir = Path('../../sdg/data/raw/')
output_dir.mkdir(parents=True, exist_ok=True)
manifest_path = output_dir / '.generation_manifest.json'

# Resume manifest
if manifest_path.exists():
    with open(manifest_path) as f:
        manifest = json.load(f)
    print(f'Resuming from manifest: completed tiers = {manifest.get("completed_tiers", [])}')
else:
    manifest = {'completed_tiers': [], 'completed_batches': {}, 'last_success_ts': None, 'total_examples': 0}

tier_order = ['single_tool', 'parallel', 'sequential', 'multi_turn', 'closed_loop']
tier_stats = {}

for tier_name in tier_order:
    if tier_name in manifest.get('completed_tiers', []):
        print(f'\nSkipping {tier_name} (already completed)')
        continue

    seeds = seeds_by_tier.get(tier_name, [])
    if not seeds:
        print(f'\nSkipping {tier_name} (no seeds)')
        continue

    print(f'\n=== Generating tier: {tier_name} ({len(seeds)} seeds) ===')

    output = flow.generate(
        seed_dataset=seeds,
        model_config=config.get('model_config', {}),
        agent_config=config.get('agent_config', {}),
        runtime_params=production_params,
    )

    # Convert to list of dicts
    if hasattr(output, 'iterrows'):
        results = [row.to_dict() for _, row in output.iterrows()]
    else:
        results = list(output)

    # Save raw output
    out_file = output_dir / f'{tier_name}_raw.jsonl'
    with open(out_file, 'w') as f:
        for r in results:
            f.write(json.dumps(r, default=str) + '\n')

    tier_stats[tier_name] = len(results)
    manifest['completed_tiers'].append(tier_name)
    manifest['total_examples'] = sum(tier_stats.values())
    manifest['last_success_ts'] = time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())

    # Atomic manifest save
    tmp_fd, tmp_path = tempfile.mkstemp(dir=str(output_dir), suffix='.tmp')
    with os.fdopen(tmp_fd, 'w') as f:
        json.dump(manifest, f, indent=2)
    os.replace(tmp_path, str(manifest_path))

    print(f'  {tier_name}: {len(results)} examples -> {out_file}')

print(f'\nTotal raw examples: {sum(tier_stats.values())}')
for tier, count in tier_stats.items():
    print(f'  {tier}: {count}')

In [ ]:
# ── Evaluation Flow and Filtering ──

eval_flow_path = FlowRegistry.get_flow_path('MCP Server Distillation Evaluation')
eval_flow = Flow.from_yaml(eval_flow_path)

filtered_dir = Path('../../sdg/data/filtered/')
filtered_dir.mkdir(parents=True, exist_ok=True)

all_filtered_rows = []
input_count = 0

for input_file in sorted(output_dir.glob('*_raw.jsonl')):
    rows = []
    with open(input_file) as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line.strip()))
    input_count += len(rows)

    if not rows:
        continue

    print(f'Evaluating {input_file.name} ({len(rows)} rows)...')
    output = eval_flow.evaluate(
        dataset=rows,
        model_config=eval_config.get('model_config', {}),
    )

    if hasattr(output, 'iterrows'):
        filtered = [row.to_dict() for _, row in output.iterrows()]
    else:
        filtered = list(output)

    # Save filtered output
    with open(filtered_dir / input_file.name, 'w') as f:
        for r in filtered:
            f.write(json.dumps(r, default=str) + '\n')

    all_filtered_rows.extend(filtered)
    print(f'  {len(filtered)}/{len(rows)} passed quality filter')

print(f'\nEvaluation: {len(all_filtered_rows)}/{input_count} rows passed')

In [ ]:
# ── Column Mapping, OSFT Validation, and Dual-Format Serialization ──
# Same pattern as run-evaluation.py: explicit column mapping, automatic
# OSFT validation (per amended D-04), and dual-format output (D-20)

from sdg.scripts.serialize_osft import serialize_single_turn, serialize_multi_turn
from sdg.lib.osft_validation import validate_osft_line, validate_osft_file

# Column mapping (same as run-evaluation.py)
COLUMN_MAP = {
    'question': 'question',
    'agent_text': 'answer',
    'response_text': 'answer',
    'tool_trace': 'tool_trace',
    'agent_tool_trace': 'tool_trace',
    'tool_list': 'tools',
}
REQUIRED_KEYS = {'question', 'tool_trace', 'answer', 'tools'}

# Apply column mapping
mapped_rows = []
for i, row in enumerate(all_filtered_rows):
    mapped = {}
    for src, dst in COLUMN_MAP.items():
        if src in row and (dst not in mapped or mapped[dst] is None):
            mapped[dst] = row[src]
    for key in REQUIRED_KEYS:
        if key in row and key not in mapped:
            mapped[key] = row[key]
    for key in ('tool_trace', 'tools'):
        if isinstance(mapped.get(key), str):
            try:
                mapped[key] = json.loads(mapped[key])
            except (json.JSONDecodeError, ValueError):
                pass
    missing = REQUIRED_KEYS - set(mapped.keys())
    if missing:
        raise ValueError(f'Row {i}: missing keys {missing}. Available: {sorted(row.keys())}')
    mapped_rows.append(mapped)

# OSFT Validation (per amended D-04 -- unconditional, automatic)
validated_rows = []
osft_errors = []
for i, row in enumerate(mapped_rows):
    try:
        serialized = serialize_single_turn(row)
    except (ValueError, KeyError) as exc:
        osft_errors.append({'row': i, 'errors': [str(exc)]})
        continue
    errors = validate_osft_line(serialized)
    if errors:
        osft_errors.append({'row': i, 'errors': errors})
    else:
        validated_rows.append(row)

print(f'OSFT validation: {len(validated_rows)}/{len(mapped_rows)} passed')
if osft_errors:
    print(f'  Dropped {len(osft_errors)} rows')

# Dual-format serialization (D-20)
delivery_dir = Path('../../sdg/data/delivery/')
delivery_dir.mkdir(parents=True, exist_ok=True)

with open(delivery_dir / 'single-turn.jsonl', 'w') as f:
    for row in validated_rows:
        f.write(json.dumps(serialize_single_turn(row)) + '\n')

with open(delivery_dir / 'multi-turn.jsonl', 'w') as f:
    for row in validated_rows:
        f.write(json.dumps(serialize_multi_turn(row)) + '\n')

# Validate delivery files
for path in [delivery_dir / 'single-turn.jsonl', delivery_dir / 'multi-turn.jsonl']:
    is_valid, errs = validate_osft_file(str(path))
    status = 'PASS' if is_valid else f'FAIL ({len(errs)} errors)'
    print(f'Delivery validation: {path.name} -> {status}')

print(f'\nDelivery files written to {delivery_dir}')

In [ ]:
# ── Final Statistics ──

# Per-tier breakdown of validated examples
tier_breakdown = {}
for row in validated_rows:
    tier = row.get('tier', 'unknown')
    tier_breakdown[tier] = tier_breakdown.get(tier, 0) + 1

print('=' * 60)
print('PRODUCTION RUN SUMMARY')
print('=' * 60)
print(f'Raw examples:         {input_count}')
print(f'Eval-filtered:        {len(all_filtered_rows)}')
print(f'OSFT-validated:       {len(validated_rows)}')
print(f'OSFT-dropped:         {len(osft_errors)}')
print(f'Final delivery count: {len(validated_rows)}')
print(f'Overall filter rate:  {(1 - len(validated_rows) / max(input_count, 1)) * 100:.1f}%')
print()
print('Per-tier distribution:')
for tier in ['single_tool', 'parallel', 'sequential', 'multi_turn', 'closed_loop']:
    count = tier_breakdown.get(tier, 0)
    print(f'  {tier}: {count}')
print()
print('Coverage report:')
# Check which tool domains are represented
domain_tools_seen = set()
for row in validated_rows:
    tool_trace = row.get('tool_trace', [])
    if isinstance(tool_trace, list):
        for t in tool_trace:
            if isinstance(t, dict):
                domain_tools_seen.add(t.get('name', 'unknown'))
print(f'  Unique tools in delivery: {len(domain_tools_seen)}')
print(f'  Tool names: {sorted(domain_tools_seen)[:20]}')
print()
print(f'Delivery files:')
print(f'  {delivery_dir / "single-turn.jsonl"}')
print(f'  {delivery_dir / "multi-turn.jsonl"}')
print('=' * 60)

## Delivery

Package the following for Farbod's team (SDG-05, D-05):

1. **Training data:** `sdg/data/delivery/single-turn.jsonl` and `sdg/data/delivery/multi-turn.jsonl`
2. **Notebooks:** `sdg/notebooks/` (this notebook + dry-run + exploration)
3. **Scripts:** `sdg/scripts/` (run-generation.py, run-evaluation.py, serialize_osft.py, materialize_seeds.py)
4. **Configs:** `sdg/config/` (flow-training.yaml, flow-evaluation.yaml, seed-scenarios.yaml)

This is the complete recipe per D-05: not just a static dataset, but the full
reproducible pipeline (notebooks + scripts + configs) that produced it.